# SESIÓN 4: CREACIÓN Y MANIPULACIÓN DE TABLAS (PARTE IV)
## Fundamentos de Programación Python para el Análisis de Datos

Eliminación de registros usando DELETE

In [ ]:
import psycopg2

# Conexión a base de datos
conn = psycopg2.connect(
    host='localhost',
    database='capacitaciones',
    user='postgres',
    password='password'
)
cur = conn.cursor()

## SLIDE 4: Sintaxis básica de DELETE

In [ ]:
# Sintaxis básica de DELETE
sql_sintaxis = """
DELETE FROM nombre_tabla
WHERE condición;
"""
print(sql_sintaxis)
print("NOTA: Si se omite WHERE, se eliminan TODOS los registros")

## SLIDE 10: Ejemplo aplicado - Crear tabla de inscripciones

In [ ]:
# Crear tabla de inscripciones
sql_create = """
DROP TABLE IF EXISTS inscripciones CASCADE;
CREATE TABLE inscripciones (
    id_inscripcion SERIAL PRIMARY KEY,
    id_estudiante INT,
    id_curso INT,
    estado VARCHAR(20)
);

INSERT INTO inscripciones (id_estudiante, id_curso, estado)
VALUES
    (1001, 501, 'Activa'),
    (1002, 502, 'Anulada'),
    (1003, 503, 'Activa'),
    (1004, 504, 'Anulada'),
    (1005, 505, 'Activa');
"""

cur.execute(sql_create)
conn.commit()
print("Tabla de inscripciones creada")

In [ ]:
# Ver datos iniciales
sql_ver = """
SELECT * FROM inscripciones
ORDER BY id_inscripcion;
"""

cur.execute(sql_ver)
registros = cur.fetchall()

print("Datos iniciales:")
print("-" * 70)
for reg in registros:
    print(f"ID: {reg[0]} | Estudiante: {reg[1]} | Curso: {reg[2]} | Estado: {reg[3]}")

## SLIDE 5: Validación previa con SELECT

In [ ]:
# BUENA PRÁCTICA: Verificar qué registros serán eliminados
sql_validar = """
SELECT * FROM inscripciones
WHERE estado = 'Anulada';
"""

cur.execute(sql_validar)
registros_anulados = cur.fetchall()

print(f"Registros a eliminar: {len(registros_anulados)}")
print("-" * 70)
for reg in registros_anulados:
    print(f"ID: {reg[0]} | Estudiante: {reg[1]} | Curso: {reg[2]} | Estado: {reg[3]}")

## SLIDE 10: Eliminación segura con DELETE

In [ ]:
# Eliminar solo las inscripciones anuladas
sql_delete = """
DELETE FROM inscripciones
WHERE estado = 'Anulada';
"""

try:
    cur.execute(sql_delete)
    conn.commit()
    print("✓ Eliminación completada")
except Exception as e:
    conn.rollback()
    print(f"✗ Error: {e}")

In [ ]:
# Verificar datos después de la eliminación
cur.execute(sql_ver)
registros = cur.fetchall()

print("Datos después de la eliminación:")
print("-" * 70)
for reg in registros:
    print(f"ID: {reg[0]} | Estudiante: {reg[1]} | Curso: {reg[2]} | Estado: {reg[3]}")
print(f"\nTotal de registros: {len(registros)}")

## SLIDE 6: Eliminación lógica vs física

In [ ]:
# BUENA PRÁCTICA: Eliminación lógica (marcar como inactivo)
# En lugar de DELETE, usar UPDATE

sql_create_logica = """
DROP TABLE IF EXISTS inscripciones_v2 CASCADE;
CREATE TABLE inscripciones_v2 (
    id_inscripcion SERIAL PRIMARY KEY,
    id_estudiante INT,
    id_curso INT,
    estado VARCHAR(20),
    activo BOOLEAN DEFAULT TRUE
);

INSERT INTO inscripciones_v2 (id_estudiante, id_curso, estado, activo)
VALUES
    (1001, 501, 'Activa', TRUE),
    (1002, 502, 'Anulada', TRUE),
    (1003, 503, 'Activa', TRUE),
    (1004, 504, 'Anulada', TRUE);
"""

cur.execute(sql_create_logica)
conn.commit()
print("Tabla con soporte para eliminación lógica creada")

In [ ]:
# Eliminación lógica: marcar como inactivo en lugar de eliminar
sql_logica = """
UPDATE inscripciones_v2
SET activo = FALSE
WHERE estado = 'Anulada';
"""

cur.execute(sql_logica)
conn.commit()
print("✓ Eliminación lógica: registros marcados como inactivos")

# Ver todos los registros (incluyendo inactivos)
sql_ver_logica = """
SELECT * FROM inscripciones_v2;
"""

cur.execute(sql_ver_logica)
registros = cur.fetchall()

print("\nTodos los registros (incluyendo inactivos):")
print("-" * 80)
for reg in registros:
    estado_visual = "✓ Activo" if reg[4] else "✗ Inactivo"
    print(f"ID: {reg[0]} | Est: {reg[1]} | Curso: {reg[2]} | Estado: {reg[3]} | {estado_visual}")

## SLIDE 11: Transacciones para reversión

In [ ]:
# Usar transacciones para poder revertir cambios
sql_trans_create = """
DROP TABLE IF EXISTS inscripciones_trans CASCADE;
CREATE TABLE inscripciones_trans (
    id_inscripcion SERIAL PRIMARY KEY,
    id_estudiante INT,
    id_curso INT,
    estado VARCHAR(20)
);

INSERT INTO inscripciones_trans (id_estudiante, id_curso, estado)
VALUES
    (1001, 501, 'Activa'),
    (1002, 502, 'Anulada'),
    (1003, 503, 'Activa'),
    (1004, 504, 'Anulada');
"""

cur.execute(sql_trans_create)
conn.commit()
print("Tabla de prueba con transacciones creada")

In [ ]:
# Demostración: Usar ROLLBACK para revertir eliminación accidental
print("Datos antes de la eliminación:")
cur.execute("SELECT COUNT(*) FROM inscripciones_trans WHERE estado = 'Activa'")
activas_antes = cur.fetchone()[0]
print(f"Inscripciones activas: {activas_antes}")

# Iniciar transacción
cur.execute("BEGIN")

# Eliminar (podría ser un error)
cur.execute("DELETE FROM inscripciones_trans WHERE estado = 'Activa'")

# Verificar qué pasaría
cur.execute("SELECT COUNT(*) FROM inscripciones_trans WHERE estado = 'Activa'")
activas_despues = cur.fetchone()[0]
print(f"Después de DELETE (sin COMMIT): {activas_despues} activas")

# ¡Oops, fue un error! Revertir con ROLLBACK
cur.execute("ROLLBACK")

# Verificar que se revirtió
cur.execute("SELECT COUNT(*) FROM inscripciones_trans WHERE estado = 'Activa'")
activas_revertidas = cur.fetchone()[0]
print(f"Después de ROLLBACK: {activas_revertidas} activas")
print("✓ Cambios revertidos correctamente")

## SLIDE 20: Actividad guiada - Eliminación segura

In [ ]:
# Crear tabla para actividad guiada
sql_actividad = """
DROP TABLE IF EXISTS inscripciones_actividad CASCADE;
CREATE TABLE inscripciones_actividad (
    id_inscripcion SERIAL PRIMARY KEY,
    id_estudiante INT,
    curso VARCHAR(50),
    estado VARCHAR(20)
);

INSERT INTO inscripciones_actividad (id_estudiante, curso, estado) 
VALUES
    (101, 'Matemáticas', 'Activa'),
    (102, 'Historia', 'Anulada'),
    (103, 'Física', 'Activa'),
    (104, 'Química', 'Anulada'),
    (105, 'Biología', 'Activa');
"""

cur.execute(sql_actividad)
conn.commit()
print("Tabla de actividad guiada creada")

In [ ]:
# Paso 1: Identificar las inscripciones anuladas
sql_identificar = """
SELECT * FROM inscripciones_actividad
WHERE estado = 'Anulada';
"""

cur.execute(sql_identificar)
anuladas = cur.fetchall()

print("Paso 1: Identificar inscripciones anuladas")
print("-" * 70)
print(f"Total encontradas: {len(anuladas)}")
for reg in anuladas:
    print(f"  ID: {reg[0]} | Estudiante: {reg[1]} | Curso: {reg[2]} | Estado: {reg[3]}")

In [ ]:
# Paso 2 & 3: Ejecutar DELETE
sql_delete_actividad = """
DELETE FROM inscripciones_actividad
WHERE estado = 'Anulada';
"""

try:
    cur.execute(sql_delete_actividad)
    conn.commit()
    print("\nPaso 2 & 3: DELETE ejecutado con éxito")
except Exception as e:
    conn.rollback()
    print(f"Error: {e}")

In [ ]:
# Paso 4: Verificar resultados
sql_verificar_final = """
SELECT * FROM inscripciones_actividad
ORDER BY id_inscripcion;
"""

cur.execute(sql_verificar_final)
restantes = cur.fetchall()

print("\nPaso 4: Datos después de la eliminación")
print("-" * 70)
print(f"Total de registros: {len(restantes)}")
for reg in restantes:
    print(f"  ID: {reg[0]} | Estudiante: {reg[1]} | Curso: {reg[2]} | Estado: {reg[3]}")

## SLIDE 26: Actividad autónoma - Eliminar duplicados

In [ ]:
# Crear tabla de asistencia con duplicados
sql_asistencia = """
DROP TABLE IF EXISTS asistencia CASCADE;
CREATE TABLE asistencia (
    id SERIAL PRIMARY KEY,
    estudiante VARCHAR(50),
    fecha DATE,
    asistencia BOOLEAN
);

INSERT INTO asistencia (estudiante, fecha, asistencia) 
VALUES
    ('Pedro', '2024-05-02', true),
    ('Pedro', '2024-05-02', true),
    ('Laura', '2024-05-02', true),
    ('Laura', '2024-05-03', false),
    ('Laura', '2024-05-03', false),
    ('Carlos', '2024-05-02', true);
"""

cur.execute(sql_asistencia)
conn.commit()
print("Tabla de asistencia con duplicados creada")

In [ ]:
# Ver datos iniciales
sql_ver_asist = """
SELECT * FROM asistencia
ORDER BY estudiante, fecha, id;
"""

cur.execute(sql_ver_asist)
registros = cur.fetchall()

print("Datos iniciales (con duplicados):")
print("-" * 70)
for reg in registros:
    asist = "Presente" if reg[3] else "Ausente"
    print(f"ID: {reg[0]} | {reg[1]} | {reg[2]} | {asist}")

In [ ]:
# Paso 1: Identificar duplicados
sql_encontrar_duplicados = """
SELECT estudiante, fecha, COUNT(*) as cantidad
FROM asistencia
GROUP BY estudiante, fecha
HAVING COUNT(*) > 1;
"""

cur.execute(sql_encontrar_duplicados)
duplicados = cur.fetchall()

print("Paso 1: Registros duplicados encontrados")
print("-" * 70)
for dup in duplicados:
    print(f"  {dup[0]} | {dup[1]} | Cantidad: {dup[2]}")

In [ ]:
# Paso 2: Eliminar duplicados (mantener solo el ID mínimo)
sql_delete_duplicados = """
DELETE FROM asistencia
WHERE id NOT IN (
    SELECT MIN(id)
    FROM asistencia
    GROUP BY estudiante, fecha
);
"""

try:
    cur.execute(sql_delete_duplicados)
    conn.commit()
    print("✓ Duplicados eliminados")
except Exception as e:
    conn.rollback()
    print(f"Error: {e}")

In [ ]:
# Paso 3: Verificar que duplicados fueron removidos
cur.execute(sql_ver_asist)
registros = cur.fetchall()

print("Datos después de eliminar duplicados:")
print("-" * 70)
for reg in registros:
    asist = "Presente" if reg[3] else "Ausente"
    print(f"ID: {reg[0]} | {reg[1]} | {reg[2]} | {asist}")
print(f"\nTotal de registros: {len(registros)}")

## Cerrar conexión

In [ ]:
cur.close()
conn.close()
print("Conexión cerrada")